#Homework 3: Machine Learning for Classification

##Dataset

В этом домашнем задании мы будем использовать набор данных Bank Marketing для оценки потенциальных клиентов.

In [ ]:
!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv

--2025-10-11 10:39:51--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.001s  

2025-10-11 10:39:51 (51.6 MB/s) - ‘course_lead_scoring.csv’ saved [80876/80876]



В этом наборе данных нашей целевой переменной для задачи классификации будет переменная 'converted', указывающая, зарегистрирован ли клиент на платформе.

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
sns.set_style("whitegrid")

##Preparing the dataset

In [ ]:
path = "/content/course_lead_scoring.csv"
df = pd.read_csv(path)
df.sample(3, random_state=42)

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
892,social_media,manufacturing,2,56070.0,self_employed,middle_east,2,0.23,1
1106,NaN,other,1,78409.0,NaN,australia,4,0.79,0
413,referral,manufacturing,2,66206.0,employed,australia,3,0.30,1


Check if the missing values are presented in the features.
If there are missing values:
- For caterogiral features, replace them with 'NA'
- For numerical features, replace with with 0.0

In [ ]:
print(df.isna().sum())
df.info()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-nul

In [ ]:
columns_categorial = ['lead_source', 'industry', 'employment_status', 'location']
df[columns_categorial] = df[columns_categorial].fillna('NA')
df['annual_income'] = df['annual_income'].fillna(0.0)
print(df.isna().sum())

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [ ]:
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NA,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NA,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


##Question 1

Какое наблюдение (мода) является наиболее частым для столбца industry?
- NA
- technology
- healthcare
- retail

In [ ]:
df['industry'].mode()

,industry
0,retail


##Question 2
Создайте матрицу корреляции для числовых признаков вашего набора данных. В матрице корреляции вычисляется коэффициент корреляции между каждой парой признаков.

Какие две характеристики имеют наибольшую корреляцию?

- interaction_count и lead_score
- number_of_courses_viewed и lead_score
- number_of_courses_viewed и interaction_count
- annual_income и interaction_count

При ответе на этот вопрос рассматривайте только указанные выше пары.

In [ ]:
pair1 = ['interaction_count', 'lead_score']
pair2 = ['number_of_courses_viewed', 'lead_score']
pair3 = ['number_of_courses_viewed', 'interaction_count']
pair4 = ['annual_income', 'interaction_count']

corr1 = df[pair1].corr()
corr2 = df[pair2].corr()
corr3 = df[pair3].corr()
corr4 = df[pair4].corr()

print(f"corr1 {corr1.iloc[0, 1]:.3f}")
print(f"corr2 {corr2.iloc[0, 1]:.3f}")
print(f"corr3 {corr3.iloc[0, 1]:.3f}")
print(f"corr4 {corr4.iloc[0, 1]:.3f}")

corr1 0.010
corr2 -0.005
corr3 -0.024
corr4 0.027


##Разделить данные
- Разделите данные на обучающие/валидационные/тестовые наборы с распределением 60%/20%/20%.
- Используйте для этого Scikit-Learn ( train_test_splitфункцию) и установите начальное значение 42.
- Убедитесь, что целевое значение yотсутствует в вашем фрейме данных.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df, test_size=0.25, random_state=42)

In [ ]:
df.shape

(1462, 9)

In [ ]:
len(df_train), len(df_val), len(df_test), len(df_train) +len(df_val)

(1096, 366, 293, 1462)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [ ]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

##Question 3

- Рассчитайте взаимную информационную оценку между 'y' и другими категориальными переменными в наборе данных. Используйте только обучающий набор.
- Округлите результаты до двух знаков после запятой, используя round(score, 2).

Какая из этих переменных имеет самый высокий показатель взаимной информации?

- industry
- location
- lead_source
- employment_status

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1462 non-null   object 
 1   industry                  1462 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1462 non-null   float64
 4   employment_status         1462 non-null   object 
 5   location                  1462 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB


In [ ]:
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']
categorical = ['lead_source', 'industry', 'employment_status', 'location']
df_full_train[categorical].nunique()

,0
lead_source,6
industry,8
employment_status,5
location,8


In [ ]:
from sklearn.metrics import mutual_info_score

In [ ]:
print(f"{mutual_info_score(df_full_train.converted, df_full_train.lead_source):.4f}")

0.0257


In [ ]:
print(f"{mutual_info_score(df_full_train.converted, df_full_train.industry):.4f}")

0.0117


In [ ]:
print(f"{mutual_info_score(df_full_train.converted, df_full_train.employment_status):.4f}")

0.0133


In [ ]:
print(f"{mutual_info_score(df_full_train.converted, df_full_train.location):.4f}")

0.0023


In [ ]:
def mutual_info_churn_score(series):
  return mutual_info_score(df_full_train.converted, series)

In [ ]:
# mi = взаимная информация
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False).round(2)

,0
lead_source,0.03
employment_status,0.01
industry,0.01
location,0.00


##Question 4

- Теперь давайте обучим логистическую регрессию.
- Помните, что в наборе данных есть несколько категориальных переменных. Включите их, используя прямое кодирование.
- Подгоните модель под обучающий набор данных.
- - Чтобы убедиться в воспроизводимости результатов в разных версиях Scikit-Learn, настройте модель с помощью следующих параметров:
- - model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Рассчитайте точность проверочного набора данных и округлите ее до 2 десятичных знаков.

Какую точность вы получили?

- 0,64
- 0,74
- 0,84
- 0,94

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
df_full_train.head(2)

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
1066,social_media,manufacturing,2,44403.0,self_employed,australia,1,0.71,0
638,events,retail,3,38048.0,student,north_america,6,0.97,1


In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [ ]:
model.intercept_[0]

np.float64(-0.07064702670205417)

In [ ]:
model.coef_[0].round(3)

array([-0.   , -0.008,  0.026,  0.006,  0.011, -0.106, -0.02 ,  0.048,
       -0.018, -0.021, -0.007, -0.006, -0.027, -0.019,  0.302,  0.047,
        0.008, -0.017, -0.014, -0.095,  0.07 , -0.023,  0.006, -0.011,
       -0.022, -0.005,  0.002,  0.003, -0.021, -0.023,  0.447])

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]
convert_decision = (y_pred >= 0.5)

In [ ]:
(convert_decision == y_val).mean()

np.float64(0.7295081967213115)

In [ ]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = convert_decision.astype(int)
df_pred['actual'] = y_val

In [ ]:
df_pred['correct'] = df_pred.prediction == df_pred.actual

In [ ]:
df_pred.correct.mean().round(2)

np.float64(0.73)

##Question 5

- Давайте найдем наименее полезную функцию, используя метод исключения функций .
- Обучите модель, используя те же признаки и параметры, что и в Q4 (без округления).
- Теперь исключите каждый признак из этого набора и обучите модель без него. Запишите точность для каждой модели.
- Для каждой характеристики рассчитайте разницу между исходной точностью и точностью без характеристики.

Какая из следующих особенностей имеет наименьшее различие?

- 'industry'
- 'employment_status'
- 'lead_score'

> Примечание : разница не обязательно должна быть положительной.

In [ ]:
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)

y_full_train = df_full_train.converted.values

In [ ]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42).fit(X_full_train, y_full_train)

In [ ]:
dicts_test = df_test[categorical + numerical].to_dict(orient='records')
X_test = dv.transform(dicts_test)

In [ ]:
y_pred = model.predict_proba(X_test)[:, 1]

In [ ]:
converted_decision = (y_pred >= 0.5)
base_accuracy = (converted_decision == y_test).mean()

In [ ]:
features = ['industry', 'employment_status', 'lead_score']
results = {}

for f in features:
    # Убираем фичу
    reduced_features = [col for col in categorical + numerical if col != f]

    # Подготовка данных без этой фичи
    dicts_full_train = df_full_train[reduced_features].to_dict(orient='records')
    dicts_test = df_test[reduced_features].to_dict(orient='records')

    # Векторизация
    dv = DictVectorizer(sparse=False)
    X_full_train = dv.fit_transform(dicts_full_train)
    X_test = dv.transform(dicts_test)

    # Обучение
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_full_train, df_full_train.converted.values)

    # Предсказания
    y_pred = model.predict_proba(X_test)[:, 1]
    acc = (y_pred >= 0.5) == y_test
    acc = acc.mean()

    # Разница
    diff = base_accuracy - acc
    results[f] = diff

results

{'industry': np.float64(0.0),
 'employment_status': np.float64(0.010238907849829282),
 'lead_score': np.float64(-0.0034129692832765013)}

##Question 6

- Теперь давайте обучим регуляризованную логистическую регрессию.
- Попробуем следующие значения параметра C: [0.01, 0.1, 1, 10, 100].
- Модели поездов, использующие все функции, как в Q4.
- Рассчитайте точность проверочного набора данных и округлите ее до трех десятичных знаков.


Какой из них Cобеспечивает наилучшую точность на проверочном наборе?

- 0,01
- 0.1
- 1
- 10
- 100

In [ ]:
C_ = [0.01, 0.1, 1, 10, 100]
results = {}

for c in C_:
  model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42).fit(X_full_train, y_full_train)

  y_pred = model.predict_proba(X_test)[:, 1]

  converted_decision = (y_pred >= 0.5)
  accuracy = (converted_decision == y_test).mean()

  results[c] = accuracy.round(3)

results

{0.01: np.float64(0.734),
 0.1: np.float64(0.741),
 1: np.float64(0.741),
 10: np.float64(0.741),
 100: np.float64(0.741)}